In [1]:
# pip install pandas ipykernel

In [ ]:
import pandas as pd
import logging
from time import sleep

from linepipe.node import Node, create_named_partial_function
from linepipe.pipeline import Pipeline

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger(__name__)

CACHE_PATH = "./.cache/features.db"

# Define your node functions

In [3]:
# Config setup, has to be dot-accesible, I use pydantic
class Config:
    def __init__(self):
        self.window = 5
        self.min_periods = 2


def load_data() -> pd.DataFrame:
    data = {
        "group_id": [1,2,3, 1,2,3, 4,4],
        "team_id": ["A", "A", "A", "B", "B", "B", "A", "C"],
        "goals": [1, 2, 0, 3, 1, 2, None, None],
    }
    return pd.DataFrame(data)


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    return df.dropna().reset_index(drop=True)


def rolling_goals(df: pd.DataFrame, window: int, min_periods: int) -> pd.DataFrame:
    df = df.copy()
    cols = df.filter(like="_id").columns.to_list()

    df["rolling_goals"] = (
        df.groupby("team_id")["goals"]
        .transform(lambda x: x.rolling(window=window, min_periods=min_periods).mean())
    )
    
    cols.append("rolling_goals")

    return df[cols]


def merge(left: pd.DataFrame, right: pd.DataFrame) -> pd.DataFrame:
    return left.merge(right, on=["group_id", "team_id"])


def write_features(df: pd.DataFrame, table_name: str) -> None:
    logger.info(f"[DB] Writing {len(df)} rows to {table_name}")

# Build Nodes

Notice what’s happening here:

* **config injection** via `config.window`
* **explicit dataset names**
* **side-effect node** with no outputs
* stable function naming for logging

In [4]:
nodes = [
    Node(
        func=load_data,
        inputs=[],
        outputs=["raw_data"],
        profile=True
    ),
    Node(
        func=clean_data,
        inputs=["raw_data"],
        outputs=["clean_data"],
    ),
    Node(
        func=rolling_goals,
        inputs=[
            "clean_data",
            "config.window",
            "config.min_periods",
        ],
        outputs=["rolling_features"]
    ),
    Node(
        func=merge,
        inputs=[
            "clean_data",
            "rolling_features",
        ],
        outputs=["features"],
        profile=True
    ),
    Node(
        func=create_named_partial_function(
            func=write_features,
            func_name="write_features",
            table_name="features",
        ),
        inputs=["features"],
        outputs=[]
    ),
]

# Run pipeline

In [5]:
pipeline = Pipeline(
    nodes=nodes,
    config=Config(),
    cache_storage_path=CACHE_PATH,
    use_persistent_cache=True,
)

pipeline.run()


INFO | linepipe.registry | Using data storage located at: .cache
INFO | linepipe.registry | Restored 6 objects from persistent cache
INFO | linepipe.pipeline | Running node: load_data
INFO | linepipe.node | [load_data] ΔMem: 0.00 MiB | Peak: 0.15 MiB
INFO | linepipe.registry | Storing output 'raw_data' in data storage.
INFO | linepipe.pipeline | Running node: clean_data
INFO | linepipe.registry | Storing output 'clean_data' in data storage.
INFO | linepipe.registry | Releasing 'raw_data' from STORAGE
INFO | linepipe.pipeline | Running node: rolling_goals
INFO | linepipe.registry | Storing output 'rolling_features' in data storage.
INFO | linepipe.pipeline | Running node: merge
INFO | linepipe.node | [merge] ΔMem: 0.00 MiB | Peak: 0.31 MiB
INFO | linepipe.registry | Storing output 'features' in data storage.
INFO | linepipe.registry | Releasing 'clean_data' from STORAGE
INFO | linepipe.registry | Releasing 'rolling_features' from STORAGE
INFO | linepipe.pipeline | Running node: write_fe

At this point, **all intermediate datasets are cached**

Even though `run()` closed the cache, you can reopen it (for example, to debug):

In [6]:
registry = pipeline.get_obj_registry()
try:
    print(registry.placement)
    print(registry.get("features").head().to_string(index=False))
finally:
    registry.close()


INFO | linepipe.registry | Using data storage located at: .cache
INFO | linepipe.registry | Restored 6 objects from persistent cache


{'raw_matches': <ObjPlacement.STORAGE: 2>, 'clean_matches': <ObjPlacement.STORAGE: 2>, 'features': <ObjPlacement.STORAGE: 2>, 'rolling_features': <ObjPlacement.STORAGE: 2>, 'raw_data': <ObjPlacement.STORAGE: 2>, 'clean_data': <ObjPlacement.STORAGE: 2>}
 group_id team_id  goals  rolling_goals
        1       A    1.0            NaN
        2       A    2.0            1.5
        3       A    0.0            1.0
        1       B    3.0            NaN
        2       B    1.0            2.0


# Restart from cache (skip expensive nodes)

Let’s say you:

* change **only** the writer logic
* or want to inspect features interactively

You can **reuse the cache**:

In [7]:
# The writer pulls `features` straight from the cache.

pipeline = Pipeline(
    nodes=[
        nodes[-1],  # only the writer node
    ],
    config=Config(),
    cache_storage_path=CACHE_PATH,
    use_persistent_cache=True,
)

pipeline.run()


INFO | linepipe.registry | Using data storage located at: .cache
INFO | linepipe.registry | Restored 6 objects from persistent cache
INFO | linepipe.pipeline | Running node: write_features
INFO | __main__ | [DB] Writing 6 rows to features
INFO | linepipe.registry | Releasing 'features' from STORAGE


# Compose pipelines

Split your pipeline logically, now you have:

* reusable feature engineering (train, predict)
* pluggable writers
* composable DAGs

In [8]:
feature_pipeline = Pipeline(
    nodes=nodes[:-1],
    config=Config(),
    use_persistent_cache=False,  # in-memory, cleaned after input is no longer needed
)

writer_pipeline = Pipeline(
    nodes=[nodes[-1]],
    config=Config(),
)

full_pipeline = feature_pipeline + writer_pipeline
full_pipeline.run()


INFO | linepipe.registry | Using in-memory cache storage
INFO | linepipe.pipeline | Running node: load_data
INFO | linepipe.node | [load_data] ΔMem: 0.00 MiB | Peak: 0.00 MiB
INFO | linepipe.registry | Storing output 'raw_data' in data storage.
INFO | linepipe.pipeline | Running node: clean_data
INFO | linepipe.registry | Storing output 'clean_data' in data storage.
INFO | linepipe.registry | Releasing 'raw_data' from STORAGE
INFO | linepipe.pipeline | Running node: rolling_goals
INFO | linepipe.registry | Storing output 'rolling_features' in data storage.
INFO | linepipe.pipeline | Running node: merge
INFO | linepipe.node | [merge] ΔMem: 0.00 MiB | Peak: 0.00 MiB
INFO | linepipe.registry | Storing output 'features' in data storage.
INFO | linepipe.registry | Releasing 'clean_data' from STORAGE
INFO | linepipe.registry | Releasing 'rolling_features' from STORAGE
INFO | linepipe.pipeline | Running node: write_features
INFO | __main__ | [DB] Writing 6 rows to features
INFO | linepipe.reg

In [9]:
from linepipe import viz

print(viz.draw_ascii_pipeline(full_pipeline))

Pipeline (5 nodes)

[load_data]
  outputs:
    - raw_data
      |
      v
[clean_data]
  inputs:
    - raw_data
  outputs:
    - clean_data
      |
      v
[rolling_goals]
  inputs:
    - clean_data
    - config.window
    - config.min_periods
  outputs:
    - rolling_features
      |
      v
[merge]
  inputs:
    - clean_data
    - rolling_features
  outputs:
    - features
      |
      v
[write_features]
  inputs:
    - features
  (no outputs)


In [10]:
viz.plot_pipeline_graph(full_pipeline)